# Week 5 — hyperparameter search for the covariate HMM

**Purpose.** One job only: search `STATES`, `AR_LAGS` and the covariate design for the
autoregressive + covariate-transition HMM on `Room 009` at 30-minute aggregation, and report the
trade-off front. Everything in `week_5.ipynb` that reports a *fitted* model — diagnostics, decoded
states, AIC/BIC tables, forecast plots, the second-order model — lives there, not here.

**Data.** The pre-built split in `{DATA_PATH}/room_009/30min`. Only `y_train` and `y_val` are read;
the test split is never touched by the search. The saved `y` is already baseline-calibrated
(min = 400 ppm), so no calibration happens here.

**Covariates are rebuilt per trial** rather than read from the saved `X`, which is frozen at week
harmonic k=1 and time-of-day k=1..3. The 30-min timestamps are reconstructed from `metadata.csv`;
rebuilding the saved design that way reproduces `X_train.csv` to ~1e-13, which is what makes the
rebuilt columns row-aligned with the saved `y`.

**Scoring.** Each trial fits Gaussian+cov, warm-starts autoregressive+cov from it, and scores the
AR model by rolling 6-hour-ahead forecasts over the validation window — held-out predictive
accuracy, not in-sample likelihood.


## Imports and config

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)   # must come before anything touches JAX

from pathlib import Path

import numpy as np
import pandas as pd
import jax.numpy as jnp
import optuna

from src.api.v4 import (
    HMM, GaussEmission, AutoregressiveGaussEmission, DynamicTransition, ForwardAlgorithm,
)
from src.base.utils import transition_matrix_to_logits
from drivers.utils import load_train_data, load_val_data, load_base_data_path


In [ ]:
# --- Search target ------------------------------------------------------
DATA_NAME = "room_009"      # {DATA_PATH}/room_009/30min: y_train / y_val / metadata
DATA_TAG  = "30min"
FIT_TOL   = 1e-2            # per-fit LL tolerance, as in the week-5 fits
N_TRIALS  = 40              # a fit pair is ~5 s, so this is a few minutes

HORIZON_HOURS = 6
BIN_SECONDS   = 1800
VAL_K = int(round(HORIZON_HOURS * 3600 / BIN_SECONDS))   # 12 steps = 6 h ahead

# Weather block: always included, never searched over. The searched part of the
# covariate design (off-day flag, time-of-day and weekly harmonics) is sampled
# per trial in `objective`.
weather_cols = [
    "mean_temp",
    "mean_relative_hum",
    "mean_wind_speed",
    "mean_pressure",
    "mean_cloud_cover",
    "mean_radiation",
]


## Covariate builder

`build_covariates(datetimes, use_off_day, tod_harmonics, week_harmonics)` returns `(X, names)`:
an `(N, D)` matrix of an optional off-day flag, cyclical week / time-of-day Fourier terms, and
hourly weather matched by time. Harmonic `k` contributes `sin/cos(2*pi*k*t / period)`.


In [ ]:
# --- Holidays -----------------------------------------------------------
try:
    import holidays as _holidays
    _dk_holidays = _holidays.Denmark()

    def _is_holiday(d):
        return d.date() in _dk_holidays
except ImportError:
    print("Warning: `holidays` package not found, using hard-coded 2024 DK holidays")
    _DK_HOLIDAYS = {
        pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-28"), pd.Timestamp("2024-03-29"),
        pd.Timestamp("2024-03-31"), pd.Timestamp("2024-04-01"), pd.Timestamp("2024-04-26"),
        pd.Timestamp("2024-05-09"), pd.Timestamp("2024-05-19"), pd.Timestamp("2024-05-20"),
        pd.Timestamp("2024-12-25"), pd.Timestamp("2024-12-26"),
    }

    def _is_holiday(d):
        return d.normalize() in _DK_HOLIDAYS

# --- Weather ------------------------------------------------------------

weather_df = pd.read_csv("data/raw/dtu/weather.csv", parse_dates=["DateFrom", "DateTo"])
weather_df = (
    weather_df[["DateFrom", *weather_cols]].dropna().sort_values("DateFrom").reset_index(drop=True)
)


def _weather_features(datetimes):
    """Match hourly weather (UTC) to each local, tz-naive observation time."""
    if not weather_cols:
        return np.empty((len(datetimes), 0))
    local = pd.DatetimeIndex(datetimes).tz_localize(
        "Europe/Copenhagen", ambiguous="NaT", nonexistent="shift_forward"
    )
    left = pd.DataFrame({"t": local.tz_convert("UTC")})
    left["order"] = np.arange(len(left))
    left = left.sort_values("t")
    merged = pd.merge_asof(
        left, weather_df.rename(columns={"DateFrom": "t"}), on="t", direction="nearest",
    )
    merged = merged.sort_values("order")
    return merged[weather_cols].ffill().bfill().to_numpy()


# --- Covariate matrix ---------------------------------------------------

def _fourier(angle, harmonics, label):
    """sin/cos pairs for each requested harmonic of a 2*pi-normalised angle."""
    cols, names = [], []
    for k in harmonics:
        cols += [np.sin(k * angle), np.cos(k * angle)]
        names += [f"sin_{label}_{k}", f"cos_{label}_{k}"]
    return cols, names


def build_covariates(datetimes, use_off_day, tod_harmonics, week_harmonics):
    """Return (X, names): an (N, D) covariate matrix and its D column names.

    The design is passed in per trial: an off-day flag, `week_harmonics` /
    `tod_harmonics` sin-cos pairs, and the `weather_cols` block, which is always
    included.
    """
    dt = pd.DatetimeIndex(datetimes)
    cols, names = [], []

    if use_off_day:
        is_weekend = dt.dayofweek >= 5
        is_holiday = np.array([_is_holiday(d) for d in dt])
        cols.append((is_weekend | is_holiday).astype(float))
        names.append("off_day")

    seconds_into_week = dt.dayofweek * 86400 + dt.hour * 3600 + dt.minute * 60 + dt.second
    c, n = _fourier(2 * np.pi * seconds_into_week / (7 * 86400), week_harmonics, "week")
    cols += c
    names += n

    seconds_into_day = dt.hour * 3600 + dt.minute * 60 + dt.second
    c, n = _fourier(2 * np.pi * seconds_into_day / 86400, tod_harmonics, "tod")
    cols += c
    names += n

    X = np.column_stack(cols) if cols else np.empty((len(dt), 0))
    return np.column_stack([X, _weather_features(datetimes)]), [*names, *weather_cols]

### Small helpers shared with `week_5.ipynb`

In [ ]:
def standardise(X, train_mask):
    mean = X[train_mask].mean(axis=0)
    std  = jnp.where(X[train_mask].std(axis=0) == 0, 1.0, X[train_mask].std(axis=0))
    return (X - mean) / std


def state_means(emission, ys):
    """Per-state base means, whichever emission is asked.

    `GaussEmission` exposes them directly as `mu`; the autoregressive emission splits
    them into `mu_vals` (state base) and `mu` (base + AR term on the previous value).
    """
    return getattr(emission, "mu_vals", emission.mu)(0, ys)



## Validation metrics and the multi-lag forecast

`rolling_forecast_ar` is the plug-in multi-step forecast, general in the number of AR lags: it
carries a window of the last `k` values (column `j` holds `y_{t-j}`, the ordering
`AutoregressiveGaussEmission.mu` expects after its flip) and pushes each prediction onto the front
of that window. The `week_5.ipynb` version reads `phi()[0]` only, so it would silently ignore lags
2+ — and this search samples `AR_LAGS` up to 6.


In [ ]:
def rmse(y_true, y_pred):
    return float(jnp.sqrt(jnp.mean((y_true - y_pred) ** 2)))

def r2_score(y_true, y_pred):
    ss_res = jnp.sum((y_true - y_pred) ** 2)
    ss_tot = jnp.sum((y_true - jnp.mean(y_true)) ** 2)
    return float(1 - ss_res / ss_tot)

def mape(y_true, y_pred):
    return float(jnp.mean(jnp.abs((y_true - y_pred) / y_true)) * 100)

In [ ]:
def rolling_forecast_ar(model, ys, xs_std, n_train, K):
    """K-step-ahead plug-in forecast anchored at every held-out observation, any AR order.

    Same structure as `rolling_discrete(..., is_ar=True)` but general in the number
    of lags: `lags[:, j]` is y_{t-j}, matching the flipped slice in
    `AutoregressiveGaussEmission.mu`, and each prediction is pushed onto the front
    of that window as the next step's lag-1 value.
    """
    ys = jnp.asarray(ys)
    T = len(ys)
    out = ForwardAlgorithm().run(model.params, model.u_pre, ys=ys, ts=None, xs=xs_std)
    utt = out.utt.reshape(T, -1)                                              # (T, S)
    # `ts` is ignored by a discrete transition, so unit waiting times are fine.
    Gammas = model.transition.transition_matrices(jnp.arange(T), jnp.ones(T), ys, xs_std)

    base = model.emission.mu_vals(0, ys, xs_std)                              # (S,) state base means
    phi = model.emission.phi()                                                # (k, S)
    k = phi.shape[0]

    anchors = jnp.arange(n_train, T - K)
    u = utt[anchors]
    lags = jnp.stack([ys[anchors - j] for j in range(k)], axis=1)              # (A, k), col 0 = newest

    # mu_s = base_s + sum_j phi_js (y_{t-j} - base_s)
    #      = base_s (1 - sum_j phi_js) + sum_j phi_js y_{t-j}
    intercept = base[None, :] * (1.0 - phi.sum(axis=0))[None, :]              # (1, S)

    y_pred = None
    for m in range(1, K + 1):
        u = jnp.einsum("ai,aij->aj", u, Gammas[anchors + m])
        mu_state = intercept + jnp.einsum("aj,js->as", lags, phi)             # (A, S)
        y_pred = jnp.sum(u * mu_state, axis=1)                                # state-weighted mean
        lags = jnp.concatenate([y_pred[:, None], lags[:, :-1]], axis=1)       # plug-in next lag

    return np.asarray(ys[anchors + K]), np.asarray(y_pred)


## The objective

In [ ]:
# =====================================================================
# Optuna search over STATES / AR_LAGS / covariate design for the
# Autoregressive + covariate HMM, scored on held-out 6h-ahead forecasts.
#
# Data: the pre-built split in {DATA_PATH}/room_009/30min. Only y_train and
# y_val are read -- the test split is never touched by the search.
# Covariates are *rebuilt* per trial (the saved X is frozen at week k=1,
# tod k=1..3) and z-scored on the training rows only.
#
# =====================================================================

_SPLIT_CACHE = {}


def load_split(data_name=DATA_NAME, tag=DATA_TAG):
    """(ys_train, ys_val, dates_tv, n_train) -- read once, then cached.

    `dates_tv` is the 30-min timestamp of every train+val row, rebuilt from
    metadata.csv. The split rows are one gap-free segment, so
    `date_range(start, periods=n_bins)` reproduces the recorded start /
    train_end / val_end exactly -- which is what keeps the rebuilt covariates
    row-aligned with the saved y.
    """
    key = (data_name, tag)
    if key not in _SPLIT_CACHE:
        ys_train, _ = load_train_data(data_name, tag)
        ys_val, _   = load_val_data(data_name, tag)
        meta = pd.read_csv(
            Path(load_base_data_path()) / data_name / tag / "metadata.csv"
        ).iloc[0]
        n_train, n_val = int(meta["n_train"]), int(meta["n_val"])
        dates = pd.date_range(meta["start"], periods=int(meta["n_bins"]), freq=meta["bin"])
        _SPLIT_CACHE[key] = (ys_train, ys_val, dates[: n_train + n_val], n_train)
    return _SPLIT_CACHE[key]


def count_free_params(model, n_frozen=1):
    """Total scalar parameters in the fitted pytree, less the frozen ones.

    `hmm_results.num_params` is `len(self.params)` -- a leaf count, not a
    scalar count -- so it cannot be used for the adjusted-R2 penalty.
    `n_frozen=1` is the frozen scalar `mu0`.
    """
    leaves = jax.tree_util.tree_leaves(model.params)
    return sum(int(np.size(l)) for l in leaves if hasattr(l, "shape")) - n_frozen


def objective(trial: optuna.Trial):
    STATES = trial.suggest_int("STATES", 2, 6)
    AR_LAGS = trial.suggest_int("AR_LAGS", 1, 6)
    USE_OFF_DAY = trial.suggest_categorical("USE_OFF_DAY", [True, False])
    TOD_HARMONICS = trial.suggest_int("TOD_HARMONICS", 0, 4)
    WEEK_HARMONICS = trial.suggest_int("WEEK_HARMONICS", 0, 3)

    # Split first: the seeds need the training series and the covariate count.
    (ys_train, ys_val), (x_train_std, x_val_std) = train_test_split_data(
        USE_OFF_DAY=USE_OFF_DAY,
        TOD_HARMONICS=TOD_HARMONICS,
        WEEK_HARMONICS=WEEK_HARMONICS,
    )
    trial.set_user_attr("n_covariates", int(x_train_std.shape[1]))

    try:
        hmm = init_hmm(STATES=STATES, num_covariates=int(x_train_std.shape[1]), ys_train=ys_train)
        hmm = train_hmm(hmm, ys_train, x_train_std, STATES=STATES, AR_LAGS=AR_LAGS)
        return validation_metrics(hmm, trial, ys_train, ys_val, x_train_std, x_val_std)
    except optuna.TrialPruned:
        raise
    except Exception as exc:                      # a diverged fit costs a trial, not the study
        trial.set_user_attr("error", f"{type(exc).__name__}: {exc}")
        raise optuna.TrialPruned(f"fit failed: {type(exc).__name__}")


def train_test_split_data(USE_OFF_DAY, TOD_HARMONICS, WEEK_HARMONICS):
    """Rebuild the covariates for this design and slice at the saved boundary.

    TOD_HARMONICS / WEEK_HARMONICS are *counts*: k means harmonics 1..k, and
    k = 0 drops that block entirely. Weather columns are always kept.
    """
    ys_train, ys_val, dates_tv, n_train = load_split()

    X_np, _ = build_covariates(
        dates_tv,
        use_off_day=USE_OFF_DAY,
        tod_harmonics=tuple(range(1, TOD_HARMONICS + 1)),
        week_harmonics=tuple(range(1, WEEK_HARMONICS + 1)),
    )
    # z-scored with training statistics only, as in the split cell
    X_std = standardise(jnp.asarray(X_np), np.arange(len(X_np)) < n_train)
    return (ys_train, ys_val), (X_std[:n_train], X_std[n_train:])


def init_hmm(STATES, num_covariates, ys_train):
    """The Gaussian + covariate HMM, seeded exactly as in the week-5 fit cell."""
    mu_seed = jnp.quantile(ys_train, jnp.linspace(0.05, 0.95, STATES)).at[0].set(450)
    sigma_seed = jnp.std(ys_train) * jnp.ones(STATES)
    tm_seed = jnp.full((STATES, STATES), 0.1).at[jnp.diag_indices(STATES)].set(0.7)
    beta_init = jnp.zeros((num_covariates, STATES, STATES - 1))

    return HMM(
        emission=GaussEmission.from_params(mu_seed, sigma_seed),
        transition=DynamicTransition(transition_matrix_to_logits(tm_seed), beta_init),
        # A DynamicTransition has no time-invariant matrix, so no stationary distribution.
        inital_distribution=jnp.full(STATES, 1.0 / STATES),
    )


def train_hmm(hmm, ys_train, x_train_std, STATES, AR_LAGS):
    """Fit the Gaussian model, warm-start the AR model from it, return the AR fit.

    At phi = 0 the AR model *is* the fitted Gaussian one, so the second fit can
    only improve on the first.
    """
    hmm.fit(ys=ys_train, ts=None, xs=x_train_std, tol=FIT_TOL, frozen={"mu0": False})

    ar_hmm = HMM(
        emission=AutoregressiveGaussEmission.from_params(
            state_means(hmm.emission, ys_train),
            jnp.exp(hmm.emission.log_sigma),
            jnp.zeros((AR_LAGS, STATES)),
        ),
        transition=DynamicTransition(hmm.transition.transition_logits, hmm.transition.beta),
        inital_distribution=jnp.full(STATES, 1.0 / STATES),
    )
    ar_hmm.fit(ys=ys_train, ts=None, xs=x_train_std, tol=FIT_TOL, frozen={"mu0": False})
    return ar_hmm


def validation_metrics(hmm, trial, ys_train, ys_val, x_train_std, x_val_std, K=VAL_K):
    """(RMSE, adjusted R2) of the rolling K-step forecast over the validation window.

    Train and validation are concatenated so a single causal forward pass reaches
    every validation anchor; filtering leaks nothing, and the covariates are
    exogenous time/weather features rather than the target.

    On a fixed validation window SST is constant, so R2 is a monotone function of
    RMSE -- the only thing adjusted R2 adds is the parameter penalty, which is what
    makes the two objectives a genuine accuracy-vs-complexity trade-off.
    """
    ys_full = jnp.concatenate([ys_train, ys_val])
    xs_full = jnp.concatenate([x_train_std, x_val_std])

    y_true, y_pred = rolling_forecast_ar(hmm, ys_full, xs_full, len(ys_train), K)
    y_true, y_pred = jnp.asarray(y_true), jnp.asarray(y_pred)

    n = len(y_true)
    p = count_free_params(hmm)
    val_rmse = rmse(y_true, y_pred)
    val_r2 = r2_score(y_true, y_pred)

    trial.set_user_attr("rmse", val_rmse)
    trial.set_user_attr("r2", val_r2)
    trial.set_user_attr("mape", mape(y_true, y_pred))
    trial.set_user_attr("log_likelihood", float(hmm.hmm_results.log_likelihood))
    trial.set_user_attr("converged", bool(hmm.hmm_results.convergence))
    trial.set_user_attr("n_params", int(p))
    trial.set_user_attr("n_val_anchors", int(n))

    if n - p - 1 <= 0:
        raise optuna.TrialPruned(f"{p} parameters vs {n} validation points: adj R2 undefined")

    adj_r2 = 1 - (1 - val_r2) * (n - 1) / (n - p - 1)
    trial.set_user_attr("adj_r2", adj_r2)

    if not (np.isfinite(val_rmse) and np.isfinite(adj_r2)):
        raise optuna.TrialPruned("non-finite validation metric")
    return val_rmse, adj_r2


## Optuna study

Multi-objective: minimise the 6 h-ahead validation RMSE and maximise adjusted R2. On a fixed
validation window R2 is a monotone function of RMSE, so the two differ only through adjusted R2's
parameter penalty — the Pareto front is an accuracy-vs-complexity trade-off, not two views of the
same thing. That also means no `study.best_params` and no report-based pruning: read
`study.best_trials`.

The SQLite storage makes a second `optimize` call **extend** this study rather than restart it;
delete `results/optuna/week5_room009.db` for a clean run.


In [ ]:
import os

os.makedirs("results/optuna", exist_ok=True)

study = optuna.create_study(
    directions=["minimize", "maximize"],          # RMSE down, adjusted R2 up
    sampler=optuna.samplers.NSGAIISampler(seed=0, population_size=16),
    study_name="week5_room009_ar_cov",
    storage="sqlite:///results/optuna/week5_room009.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

trials_df = study.trials_dataframe()
trials_df


In [ ]:
print(f"{len(study.trials)} trials, {len(study.best_trials)} on the Pareto front\n")

front = sorted(study.best_trials, key=lambda t: t.values[0])
for t in front:
    print(f"trial {t.number:>3}  RMSE {t.values[0]:8.2f}  adjR2 {t.values[1]:+.4f}  "
          f"(R2 {t.user_attrs['r2']:+.4f}, p={t.user_attrs['n_params']:>4}, "
          f"D={t.user_attrs['n_covariates']:>2})  {t.params}")

best = front[0]
print(f"\nLowest-RMSE front member: trial {best.number} -> {best.params}")
print("  NOTE: this is one end of the front, not 'the' optimum -- the other end trades")
print("  RMSE for fewer parameters. Pick with the report's purpose in mind.")


## Notes

- Each harmonic adds 2 covariates and therefore `2 * STATES * (STATES - 1)` `beta` parameters, so
  `TOD_HARMONICS` and `STATES` interact strongly in the penalty. At `STATES = 6` with the full
  design, `beta` alone approaches the ~625 validation anchors and adjusted R2 goes negative — the
  penalty judging that corner of the space unusable rather than merely worse.
- The front is a trade-off, not a ranking: its low-RMSE end buys a few percent of accuracy with
  three times the parameters. Pick the member that suits what the result is for, and refit it in
  `week_5.ipynb` to get the diagnostics and the (still untouched) test-set numbers.
